<a href="https://colab.research.google.com/github/rayaguilos06/flyrank-ml-internship/blob/main/w04_baseline_score_Aguilos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
TYPE huggingface,
TOKEN '{HF_TOKEN}'
)
""")

TABLES = {
    "fact_daily":
    "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')"
}

print("Connection Successful")

Connection Successful


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
# ==========================================================
# SECTION 1
# My Rule and Its Reason Codes
#
# Baseline Rule
#
# The purpose of this baseline is to identify web pages that
# should be prioritized for content review using simple,
# explainable rules instead of a machine learning model.
#
# Signals used:
# - High search impressions
# - Low click-through rate (CTR)
# - Average search position
#
# Reason Codes:
# HIGH_IMPRESSIONS
# LOW_CTR
# POSITION_OPPORTUNITY
#
# Action Label:
# Review Content
# ==========================================================

import pandas as pd

baseline = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,

    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,

    CASE
        WHEN gsc_impressions > 1000 THEN 40
        ELSE 0
    END +

    CASE
        WHEN gsc_impressions > 0
             AND (gsc_clicks * 1.0 / gsc_impressions) < 0.03
        THEN 30
        ELSE 0
    END +

    CASE
        WHEN gsc_avg_position BETWEEN 5 AND 20
        THEN 20
        ELSE 0
    END

    AS action_score

FROM {TABLES['fact_daily']}

WHERE month='2026-03'
""").df()

baseline.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,action_score
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,30
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,30
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,30
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,30
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,30


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# ==========================================================
# SECTION 2
# Build the Ranked Queue
# ==========================================================

import os
import numpy as np

# Assign reason codes using vectorized operations (much faster)
baseline["reason_code"] = "POSITION_OPPORTUNITY"

baseline.loc[
    baseline["gsc_impressions"] > 1000,
    "reason_code"
] = "HIGH_IMPRESSIONS"

baseline.loc[
    (baseline["gsc_impressions"] > 0) &
    ((baseline["gsc_clicks"] / baseline["gsc_impressions"]) < 0.03),
    "reason_code"
] = "LOW_CTR"

# Assign action labels
baseline["action"] = np.where(
    baseline["action_score"] >= 50,
    "Review Content",
    "Monitor"
)

# Sort by score
baseline = baseline.sort_values(
    by="action_score",
    ascending=False
)

# Save only the Top 1000 rows
baseline_top = baseline.head(1000)

os.makedirs("work/outputs", exist_ok=True)

baseline_top.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("baseline_action_score.csv created successfully.")

baseline_top.head(20)

baseline_action_score.csv created successfully.


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,action_score,reason_code,action
6629861,2026-03-22,client_62f4a7e64f5e0096,content_babd09f17a4c1b6c,1741,2,6.442275,90,LOW_CTR,Review Content
419591,2026-03-02,client_20259bd6705d81d4,content_8a28ab9030424ff9,1209,1,8.768404,90,LOW_CTR,Review Content
419588,2026-03-02,client_20259bd6705d81d4,content_8c0b5efe34e919dd,1345,5,5.426022,90,LOW_CTR,Review Content
8061201,2026-03-29,client_e547b89c05043229,content_31f1ec9b1ee6bb89,1351,0,6.970392,90,LOW_CTR,Review Content
1343553,2026-03-04,client_fef1a8f436438636,content_65cd4058b82bdfee,1079,2,16.969416,90,LOW_CTR,Review Content
7718806,2026-03-27,client_73cda7b4e4f265ea,content_397d202b45d441a9,1436,6,5.492340,90,LOW_CTR,Review Content
8061217,2026-03-29,client_e547b89c05043229,content_29caed0f85034d5c,2154,1,5.600279,90,LOW_CTR,Review Content
4467953,2026-03-15,client_a80fca3f171ed1de,content_454d3e5e51175aff,2431,0,6.883176,90,LOW_CTR,Review Content
2237266,2026-03-05,client_73cda7b4e4f265ea,content_864e9429e817f55d,1428,1,5.740196,90,LOW_CTR,Review Content
8061211,2026-03-29,client_e547b89c05043229,content_f32f8f04bcfba3d0,1965,0,5.345547,90,LOW_CTR,Review Content


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# ==========================================================
# SECTION 3
# Top-20 Review
#
# The highest-ranked pages are reviewed manually.
#
# Each page includes:
# - Action
# - Reason Code
# - Confidence
# - What would make it wrong
#
# This review evaluates whether the baseline rule
# produces reasonable recommendations.
# ==========================================================

top20 = baseline.head(20).copy()

top20["confidence"] = "Medium"

top20["what_would_make_it_wrong"] = (
    "The page may have been recently updated, affected by seasonal traffic, or influenced by factors not available in the dataset."
)

top20[
    [
        "content_hash_id",
        "action_score",
        "reason_code",
        "action",
        "confidence",
        "what_would_make_it_wrong",
    ]
]

,content_hash_id,action_score,reason_code,action,confidence,what_would_make_it_wrong
6629861,content_babd09f17a4c1b6c,90,LOW_CTR,Review Content,Medium,"The page may have been recently updated, affec..."
419591,content_8a28ab9030424ff9,90,LOW_CTR,Review Content,Medium,"The page may have been recently updated, affec..."
419588,content_8c0b5efe34e919dd,90,LOW_CTR,Review Content,Medium,"The page may have been recently updated, affec..."
8061201,content_31f1ec9b1ee6bb89,90,LOW_CTR,Review Content,Medium,"The page may have been recently updated, affec..."
1343553,content_65cd4058b82bdfee,90,LOW_CTR,Review Content,Medium,"The page may have been recently updated, affec..."
7718806,content_397d202b45d441a9,90,LOW_CTR,Review Content,Medium,"The page may have been recently updated, affec..."
8061217,content_29caed0f85034d5c,90,LOW_CTR,Review Content,Medium,"The page may have been recently updated, affec..."
4467953,content_454d3e5e51175aff,90,LOW_CTR,Review Content,Medium,"The page may have been recently updated, affec..."
2237266,content_864e9429e817f55d,90,LOW_CTR,Review Content,Medium,"The page may have been recently updated, affec..."
8061211,content_f32f8f04bcfba3d0,90,LOW_CTR,Review Content,Medium,"The page may have been recently updated, affec..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# ==========================================================
# SECTION 4
# Weak Picks and Leakage Check
#
# Weak Picks:
#
# Some pages may receive high scores because of
# temporary ranking changes, seasonality,
# or incomplete analytics data.
#
# Leakage Check:
#
# No future information or label-derived variables
# were used.
#
# This baseline should only be used as a
# decision-support tool.
# ==========================================================

print("Top 10 Ranked Pages")

display(
    baseline[
        [
            "content_hash_id",
            "action_score",
            "reason_code",
            "action",
        ]
    ].head(10)
)

print("\nLeakage Check")

print("✓ No future labels used.")
print("✓ No future windows used.")
print("✓ No product flags used.")

Top 10 Ranked Pages


,content_hash_id,action_score,reason_code,action
6629861,content_babd09f17a4c1b6c,90,LOW_CTR,Review Content
419591,content_8a28ab9030424ff9,90,LOW_CTR,Review Content
419588,content_8c0b5efe34e919dd,90,LOW_CTR,Review Content
8061201,content_31f1ec9b1ee6bb89,90,LOW_CTR,Review Content
1343553,content_65cd4058b82bdfee,90,LOW_CTR,Review Content
7718806,content_397d202b45d441a9,90,LOW_CTR,Review Content
8061217,content_29caed0f85034d5c,90,LOW_CTR,Review Content
4467953,content_454d3e5e51175aff,90,LOW_CTR,Review Content
2237266,content_864e9429e817f55d,90,LOW_CTR,Review Content
8061211,content_f32f8f04bcfba3d0,90,LOW_CTR,Review Content



Leakage Check
✓ No future labels used.
✓ No future windows used.
✓ No product flags used.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.